# Combined Lab: Standard + Streaming Language Modeling Pipeline

This lab combines the ideas from the two provided notebooks into one end-to-end workflow.

## What is different here?
- **Dataset:** `roneneldan/TinyStories` instead of WikiText
- **Model / Tokenizer:** `distilgpt2` (free and readily available on Hugging Face)
- **Coverage:** both
  1. a regular in-memory preprocessing pipeline, and  
  2. a true streaming preprocessing pipeline

## Goal
Build PyTorch-ready batches for causal language modeling using Hugging Face `datasets` and `transformers`.

In [ ]:
# If needed, install the dependencies:
# !pip install datasets transformers torch accelerate -q

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import DataLoader, IterableDataset
import torch

## 1. Choose a free model and dataset

We use:
- **Model:** `distilgpt2`
- **Dataset:** `roneneldan/TinyStories`

`distilgpt2` is a lightweight open model that is easy to download and use for teaching/demo purposes.

In [ ]:
model_name = "distilgpt2"
dataset_name = "roneneldan/TinyStories"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-style tokenizers usually do not define a pad token

# Optional sanity check: load the model config/weights
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model loaded:", model.__class__.__name__)

## 2. Regular in-memory pipeline

This section mirrors the first notebook:
1. load the dataset normally,
2. tokenize with `.map(...)`,
3. concatenate and chunk into fixed-length blocks,
4. build a `DataLoader`.

In [ ]:
dataset = load_dataset(dataset_name, split="train[:1%]")  # small subset for quick experimentation
print("Number of examples:", len(dataset))
print(dataset[0].keys())
print(dataset[0]["text"][:200])

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["text"], return_special_tokens_mask=False)

tokenized_ds = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
print(tokenized_ds[0]["input_ids"][:20])

In [ ]:
block_size = 128

def group_texts(examples):
    concatenated = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = (len(concatenated["input_ids"]) // block_size) * block_size

    result = {
        k: [t[i:i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated.items()
    }
    return result

lm_ds = tokenized_ds.map(group_texts, batched=True, batch_size=1000)
print("Number of LM chunks:", len(lm_ds))
print("First chunk length:", len(lm_ds[0]["input_ids"]))

In [ ]:
def collate_fixed(batch):
    input_ids = torch.tensor([example["input_ids"] for example in batch], dtype=torch.long)
    attention_mask = torch.tensor([example["attention_mask"] for example in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.clone()
    }

train_loader = DataLoader(lm_ds, batch_size=8, shuffle=True, collate_fn=collate_fixed)

batch = next(iter(train_loader))
print("Regular pipeline batch shapes:")
for k, v in batch.items():
    print(k, v.shape)

## 3. True streaming pipeline

This section mirrors the second notebook:
1. load the dataset with `streaming=True`,
2. tokenize lazily,
3. keep a rolling token buffer,
4. emit fixed-size language-model chunks on the fly.

In [ ]:
stream_dataset = load_dataset(dataset_name, split="train", streaming=True)
print(stream_dataset)

In [ ]:
def tokenize_stream_batch(examples):
    return tokenizer(examples["text"])

tokenized_stream = stream_dataset.map(tokenize_stream_batch, batched=True)

In [ ]:
def group_texts_streaming(dataset_iter, block_size):
    buffer = []

    for example in dataset_iter:
        if "input_ids" not in example:
            continue

        buffer.extend(example["input_ids"])

        while len(buffer) >= block_size:
            chunk = buffer[:block_size]
            buffer = buffer[block_size:]

            yield {
                "input_ids": chunk,
                "attention_mask": [1] * block_size
            }

In [ ]:
class StreamingLMIterableDataset(IterableDataset):
    def __init__(self, hf_iterable_dataset, block_size):
        self.dataset = hf_iterable_dataset
        self.block_size = block_size

    def __iter__(self):
        return group_texts_streaming(iter(self.dataset), self.block_size)

streaming_lm_ds = StreamingLMIterableDataset(tokenized_stream, block_size)

In [ ]:
def collate_stream(batch):
    input_ids = torch.tensor([x["input_ids"] for x in batch], dtype=torch.long)
    attention_mask = torch.tensor([x["attention_mask"] for x in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.clone()
    }

stream_loader = DataLoader(streaming_lm_ds, batch_size=8, collate_fn=collate_stream)

In [ ]:
print("Sample streaming batches:")
for i, batch in enumerate(stream_loader):
    print(f"Batch {i}")
    for k, v in batch.items():
        print(" ", k, v.shape)
    if i == 2:
        break

## 4. Optional forward pass sanity check

This is only to confirm the batch format works with the model.

In [ ]:
sample_batch = next(iter(train_loader))

with torch.no_grad():
    outputs = model(
        input_ids=sample_batch["input_ids"],
        attention_mask=sample_batch["attention_mask"],
        labels=sample_batch["labels"]
    )

print("Loss:", float(outputs.loss))
print("Logits shape:", outputs.logits.shape)

## 5. Summary

This combined notebook shows two common LM data-pipeline styles:

- **Regular pipeline**
  - easy to inspect,
  - good for smaller datasets,
  - uses `.map(...)` and precomputed chunks.

- **Streaming pipeline**
  - memory efficient,
  - better for large datasets,
  - tokenizes and chunks lazily.

Both produce batches of:
- `input_ids`
- `attention_mask`
- `labels`

which are ready for causal language-model training.